In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import hvplot.pandas
import holoviews as hv
from workspace_dani.data_preprocessing.utils import data_handling
import numpy as np


hv.extension("bokeh")


In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("build") / Path(
    "2024-03-26") / Path("build_dataset_cats.parquet")
dataset_path

In [ ]:
df_patients = data_handling.load_parquet_table(parquet_path=dataset_path, debug_mode=False, backend="numpy_nullable")

In [ ]:
df_patients.head()

In [ ]:
df_unit_discharge_offsets = df_patients[["patientunitstayid", "unitdischargeoffset"]]

In [ ]:
#df_unit_discharge_offsets["unitdischargeoffset_min"] = (
df_unit_discharge_offsets.insert(2,value=pd.to_timedelta(df_unit_discharge_offsets.loc[:,("unitdischargeoffset")], unit="min"), column="unitdischargeoffset_min")

In [ ]:
df_unit_discharge_offsets.loc[0:9, "unitdischargeoffset"]

In [ ]:
df_unit_discharge_offsets.insert(3, value=df_unit_discharge_offsets.loc[:,"unitdischargeoffset_min"] / np.timedelta64(1, "h"), column="unitdischargeoffset_hours")

In [ ]:
df_unit_discharge_offsets["unitdischargeoffset"].hvplot("unitdischargeoffset", kind="hist", title="Unit Discharge Offset")

In [ ]:
df_unit_discharge_offsets[df_unit_discharge_offsets["unitdischargeoffset_hours"] < 2000]["unitdischargeoffset_hours"].hvplot("unitdischargeoffset_hours", kind="hist", title="Unit Discharge Offset", bins=30)


### Getting the patients that fall within our cohort time constraints

We will exclude patients with "too short" ICU stays since, we assume that there's not enough information to work with to make a valid predictions since we wan't to include variables that are related to e.g. lab measurements.
Furthermore, we will exclude patients with stays longer than 192 h due to the fact that we want to use the most typical ICU stay type, we assume patient with stays longer than 8 days to be complicated cases that shouldn't skew our data.
On top of that, we can compare our result better to the moor paper this way!


In [ ]:
# those are in hours
unit_discharge_lower_bound: int = 6
unit_discharge_upper_bound: int = 192


In [ ]:
lower_offset_condition = df_unit_discharge_offsets["unitdischargeoffset_hours"] >= unit_discharge_lower_bound
upper_offset_condition = df_unit_discharge_offsets["unitdischargeoffset_hours"] <= unit_discharge_upper_bound

In [ ]:
df_unit_discharge_offsets_cohorts = df_unit_discharge_offsets.loc[lower_offset_condition & upper_offset_condition]

#### Sample number reduction:

In [ ]:
less_samples: int = len(df_unit_discharge_offsets["patientunitstayid"]) - len(df_unit_discharge_offsets_cohorts["patientunitstayid"])

In [ ]:
less_samples

In [ ]:
len(df_unit_discharge_offsets["patientunitstayid"]) / less_samples

In [ ]:
df_unit_discharge_offsets_cohorts["unitdischargeoffset_hours"].hvplot("unitdischargeoffset_hours", kind="hist", title="Unit Discharge Offset")


In [ ]:
patient_ids_dir: Path = project_root_path / Path("intermediary_data") / Path("patient_unit_stay_ids") /  Path("cohorts") / Path("unit_discharge_offset")

In [ ]:
ids_included = df_unit_discharge_offsets_cohorts["patientunitstayid"].unique().tolist()
df_included_ids: pd.DataFrame = pd.DataFrame(data=ids_included, columns=["Included ID"])

In [ ]:

df_included_ids.to_parquet(patient_ids_dir / Path("included_ids_unit_discharge_offset.parquet") , index=False)